# delta-mem adapter fine-tune on Kaggle T4

Trains the `declare-lab/delta-mem_qwen3_4b-instruct` adapter at 4-8 k context on a single Kaggle T4 (16 GB).

**Settings**: Accelerator = `GPU T4 x1` · Internet = ON · Persistence = Variables and files

**See** [`KAGGLE_INSTRUCTIONS.md`](KAGGLE_INSTRUCTIONS.md) in the repo for full context and the T4 x2 variant.

## Cell 1 — Environment + wheel cache restore

Restores cached wheels and Triton JIT cache from a Kaggle Dataset if one is attached (named anything containing `wheel_cache` under `/kaggle/input/`). On a fresh session with no cache attached, this is a ~1 s no-op.

In [ ]:
import os, shutil, subprocess
from pathlib import Path

print('--- gpu ---')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                       '--format=csv'], capture_output=True, text=True).stdout)

# Restore cached wheels + Triton kernels from an attached dataset if present
for entry in Path('/kaggle/input').iterdir() if Path('/kaggle/input').exists() else []:
    if 'wheel_cache' in entry.name.lower():
        pip_src = entry / 'pip'
        triton_src = entry / 'triton'
        if pip_src.exists():
            shutil.copytree(pip_src, '/root/.cache/pip', dirs_exist_ok=True)
            print(f'restored pip cache from {entry}')
        if triton_src.exists():
            shutil.copytree(triton_src, '/root/.triton/cache', dirs_exist_ok=True)
            print(f'restored Triton kernel cache from {entry}')
        break
else:
    print('no wheel_cache dataset attached — first-time install path')

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'   # faster HF downloads
os.environ['TOKENIZERS_PARALLELISM'] = 'false'  # avoid warnings

## Cell 2 — Install dependencies

Kaggle's base image already ships PyTorch + transformers + Triton, but versions may not match what our submodules need. Pin to the same versions as the local env.

In [ ]:
import subprocess, sys

# Required packages with versions matching .venv on the local CUDA host.
# Kaggle has most of these; pip will skip what's already there.
PACKAGES = [
    'torch==2.5.1',           # Kaggle T4 base image usually has compatible torch
    'transformers==5.9.0',
    'accelerate>=0.34',
    'peft>=0.14',
    'safetensors',
    'huggingface_hub[hf_transfer]',
    'datasets',
    'bitsandbytes>=0.43',     # 8-bit Adam if we go memory-tight
    'triton>=3.1',            # delta-mem scan kernel
]

subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                        '--cache-dir', '/root/.cache/pip',
                        '--quiet'] + PACKAGES)
print('install complete')

# Verify the critical imports work before going further
import torch, transformers, triton
print(f'torch={torch.__version__}  cuda={torch.cuda.is_available()}  '
      f'transformers={transformers.__version__}  triton={triton.__version__}')

## Cell 3 — Clone the repo (shallow, with submodules)

Shallow clone keeps this under a minute. The two submodules (`delta-Mem`, `oscar-transformers`) need to be initialised.

In [ ]:
import subprocess, os
from pathlib import Path

REPO = Path('/kaggle/working/delta-mem-tests')
if not REPO.exists():
    subprocess.check_call([
        'git', 'clone', '--depth', '1', '--shallow-submodules',
        '--recurse-submodules',
        'https://github.com/jamesburton/delta-mem-tests.git',
        str(REPO),
    ])
else:
    subprocess.check_call(['git', '-C', str(REPO), 'pull', '--rebase'])
    subprocess.check_call(['git', '-C', str(REPO), 'submodule', 'update',
                            '--init', '--recursive'])

# Install our submodule's editable package and the delta-Mem package
subprocess.check_call(['pip', 'install', '--cache-dir', '/root/.cache/pip',
                        '--quiet', '-e', str(REPO / 'third_party/oscar-transformers')])
subprocess.check_call(['pip', 'install', '--cache-dir', '/root/.cache/pip',
                        '--quiet', '-e', str(REPO / 'delta-Mem')])

os.chdir(REPO)
print(f'repo at {REPO}')
print(subprocess.check_output(['git', 'log', '--oneline', '-3'], text=True))

## Cell 4 — Training smoke (REQUIRED gate)

Runs the 6-check smoke at 256 tokens. If this fails, **stop** and inspect the error rather than spending GPU hours on a broken pipeline. The same smoke caught two real bugs locally (adapter loader ordering and gradient checkpointing reentrant flag).

In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, '-m', 'run.training_smoke'],
                          capture_output=False)
if result.returncode != 0:
    raise RuntimeError(f'training smoke failed (rc={result.returncode}) — stop here')
print('\nsmoke OK — safe to continue to training')

## Cell 5 — Train the adapter

T4 budget: 8 k context is the natural target (tight at ~15.8 GB peak). For safer headroom on first run, start at 4 k.

Wall-time estimates (no Windows paging penalty):
- 4 k context, 200 steps: ~20-30 min
- 8 k context, 200 steps: ~45-70 min  
- 8 k context, 1000 steps: ~4-6 h (within the 9 h continuous-compute cap)

In [ ]:
import subprocess, sys
from pathlib import Path

OUT = Path('/kaggle/working/checkpoints/lora_v0_kaggle_t4')
OUT.parent.mkdir(parents=True, exist_ok=True)

# Start safe at 4 k context. Move to 8 k once you've seen a clean 4 k run.
CONTEXT = 4096       # bump to 8192 after first clean run
STEPS = 200          # bump to 500-1000 for a real fine-tune
LR = 1e-5

result = subprocess.run([
    sys.executable, '-m', 'run.local_lora_train',
    '--steps', str(STEPS),
    '--context', str(CONTEXT),
    '--lr', str(LR),
    '--out', str(OUT),
], capture_output=False)
if result.returncode != 0:
    raise RuntimeError(f'training failed (rc={result.returncode})')

print(f'\ncheckpoint saved to {OUT}')
for f in OUT.iterdir():
    print(f'  {f.name:>35} {f.stat().st_size/1024:.1f} KB')

## Cell 6 — Round-trip eval (optional, 3 q smoke)

Validates the trained checkpoint by running it through the full eval pipeline via `--adapter-override`. 3 questions is enough to confirm the load path works; for a real quality signal, run 10 q here or back on the local host.

In [ ]:
import os, subprocess, sys
from pathlib import Path

# OSCAR backend needs the calibrated rotations. The repo already has them under data/oscar/.
# If they're absent (rare on a fresh clone), download them via the existing helper.
rot_k = Path('data/oscar/rotations/instruct_gpqa/k_rotation_qqt_r_h_pbr.pt')
rot_v = Path('data/oscar/rotations/instruct_gpqa/v_rotation_sst_r_h_pbr.pt')
if not (rot_k.exists() and rot_v.exists()):
    print('OSCAR rotations not in repo — skipping eval. Run on local host instead.')
else:
    env = os.environ.copy()
    env.update({
        'KV_CACHE_BACKEND': 'oscar',
        'KV_CACHE_BITS': '2',
        'OSCAR_K_ROTATION_PATH': str(rot_k),
        'OSCAR_V_ROTATION_PATH': str(rot_v),
    })
    result = subprocess.run([
        sys.executable, '-m', 'run.locomo_eval',
        '--kv-cache-backend', 'oscar', '--kv-cache-bits', '2',
        '--max-conversations', '1', '--max-questions-per-conversation', '3',
        '--adapter-override', '/kaggle/working/checkpoints/lora_v0_kaggle_t4',
        '--output-json', '/kaggle/working/lora_v0_kaggle_t4_eval.json',
    ], env=env, capture_output=False)
    print('\neval rc=', result.returncode)

## Cell 7 — Cache snapshot + checkpoint zip for download

Run this last. It bundles the pip + Triton caches into `/kaggle/working/wheel_cache/` so you can save them as a new version of your `wheel_cache` dataset for next session, and zips the checkpoint so it's easy to download from the notebook output panel.

In [ ]:
import shutil
from pathlib import Path

CACHE_DEST = Path('/kaggle/working/wheel_cache')
CACHE_DEST.mkdir(exist_ok=True)
if Path('/root/.cache/pip').exists():
    shutil.copytree('/root/.cache/pip', CACHE_DEST / 'pip', dirs_exist_ok=True)
if Path('/root/.triton/cache').exists():
    shutil.copytree('/root/.triton/cache', CACHE_DEST / 'triton', dirs_exist_ok=True)
print(f'wheel + Triton cache snapshot at {CACHE_DEST}')
print(f'  -> save as a new version of your wheel_cache dataset')

ckpt = Path('/kaggle/working/checkpoints/lora_v0_kaggle_t4')
if ckpt.exists():
    archive = shutil.make_archive('/kaggle/working/checkpoint',
                                    'zip', root_dir=ckpt.parent, base_dir=ckpt.name)
    print(f'\ncheckpoint archived to {archive}')
    print('  -> download from the notebook "Output" panel after Save Version')